In [140]:
import pandas as pd

In [141]:
# read .dta file using pandas
censo_coord_df = pd.read_stata("data/CEN2023_VIVIENDASconPERSONAS_LATLON.dta")
censo_persona_df = pd.read_stata("data/PERSONA.dta")
censo_hogar_df = pd.read_stata("data/HOGAR.dta")

In [142]:
# Create a copy to avoid SettingWithCopyWarning when adding new columns
personas_df = censo_persona_df[['p03_edad','p14_escu', 'p02_sexo', 'p14_asistio', 'pea_nea', 'p18_ocup_cod4', 'ingr_per', 'p17e_motivo', 'llaveviv', 'hogar', 'npersona', 'provincia', 'distrito', 'correg', 'area', 'lugar_pob', 'barriada', 'p223_jubi']].copy()

In [143]:
# Convert ingr_per from category to integer
personas_df["ingr_per"] = pd.to_numeric(personas_df["ingr_per"], errors='coerce').fillna(0).astype(int)
personas_df["p223_jubi"] = pd.to_numeric(personas_df["p223_jubi"], errors='coerce').fillna(0).astype(int)

In [144]:
personas_df["p02_sexo"].describe()

count     4064780
unique          2
top         Mujer
freq      2049962
Name: p02_sexo, dtype: object

In [145]:
# Create dummy variable by economic activity 
#workers
personas_df["worker"] = personas_df["pea_nea"] == "Ocupada"
#unemployed 
personas_df["unemployed"] = personas_df["pea_nea"] == "Desocupada"
# sududy and work
personas_df["student_worker"] = (personas_df["p14_escu"] == 'S�') & (personas_df["worker"] == True)
#students
personas_df["student"] = (personas_df["p14_escu"] == 'S�') & (personas_df["worker"] == False)
print(personas_df["student"].unique())
#retired
personas_df["retired"] = personas_df["p17e_motivo"] == 'Jubilado o pensionado?'
print(personas_df["retired"].unique())

#inactive: "No económicamente activa" but NOT a student and NOT retired
personas_df["inactive"] = (
    ((personas_df["p03_edad"] < 10) & (personas_df["student"] == False)) |
    (personas_df["pea_nea"] == "No econ�micamente activa") & (personas_df["student"] == False) & (personas_df["retired"] == False))
# Types of workers - by income (ONLY for workers)
# Initialize income variables to 0 for everyone
personas_df["low_income_worker"] = 0
personas_df["high_income_worker"] = 0
personas_df["pension_retiree"] = 0
personas_df["no_pension_retiree"] = 0

# Calculate median income of the work population
median_income_workers = personas_df[personas_df["worker"] == 1]["ingr_per"].median()
# Classify ONLY workers into lower income and higher income
worker_mask = personas_df["worker"] == 1
personas_df.loc[worker_mask, "low_income_worker"] = (personas_df.loc[worker_mask, "ingr_per"] < median_income_workers).astype(int)
personas_df.loc[worker_mask, "high_income_worker"] = (personas_df.loc[worker_mask, "ingr_per"] >= median_income_workers).astype(int)

# Types of retirees - by pension status (ONLY for retirees)
# Create pension_retiree dummy: 1 if retiree has pension income (p223_jubi > 0), 0 otherwise
retiree_mask = personas_df["retired"] == 1
personas_df.loc[retiree_mask, "pension_retiree"] = (personas_df.loc[retiree_mask, "p223_jubi"] > 0).astype(int)
personas_df.loc[retiree_mask, "no_pension_retiree"] = (personas_df.loc[retiree_mask, "p223_jubi"] == 0).astype(int)


personas_df["age"] = personas_df["p03_edad"]
personas_df["student_worker"] = personas_df["student_worker"].astype(int)
personas_df["worker"] = personas_df["worker"].astype(int)
personas_df["unemployed"] = personas_df["unemployed"].astype(int)
personas_df["inactive"] = personas_df["inactive"].astype(int)
personas_df["student"] = personas_df["student"].astype(int)
personas_df["retired"] = personas_df["retired"].astype(int)
personas_df["pension_retiree"] = personas_df["pension_retiree"].astype(int)
personas_df["no_pension_retiree"] = personas_df["no_pension_retiree"].astype(int)



[False  True]
[ True False]


In [146]:
# Create cotus: Sex and HH income 
personas_df["sexo"] = personas_df["p02_sexo"]
censo_hogar_df["ingr_hog"] = censo_hogar_df["ingr_hog"]
# Convert column names to lowercase
censo_coord_df.columns = censo_coord_df.columns.str.lower()

In [147]:
print(personas_df.columns)
print(censo_hogar_df.columns)
print(censo_coord_df.columns)


Index(['p03_edad', 'p14_escu', 'p02_sexo', 'p14_asistio', 'pea_nea',
       'p18_ocup_cod4', 'ingr_per', 'p17e_motivo', 'llaveviv', 'hogar',
       'npersona', 'provincia', 'distrito', 'correg', 'area', 'lugar_pob',
       'barriada', 'p223_jubi', 'worker', 'unemployed', 'student_worker',
       'student', 'retired', 'inactive', 'low_income_worker',
       'high_income_worker', 'pension_retiree', 'no_pension_retiree', 'age',
       'sexo'],
      dtype='object')
Index(['llaveviv', 'hogar', 'provincia', 'distrito', 'correg', 'area',
       'lugar_pob', 'barriada', 'h18a_estu', 'h18b_refr', 'h18c_lava',
       'h18d_mcos', 'h18e_aban', 'h18f_aire', 'h18g_radi', 'h18h_tres',
       'h18i_tcel', 'h18j_tv', 'h18j_cable', 'h18k_comp', 'h18l_inter',
       'h18m_auto', 'h19_opai', 'h19_cuantos', 'h20_covid', 'h20_covid_cant',
       'h201_fallecio', 'h201_cuantos', 'h21a_cultivo', 'h21b_cria',
       'h21c_activ', 'lo_peradic', 'h_personas', 'h_hombres', 'h_mujeres',
       'hv01_tipo', 'hrv0

In [148]:
# merge censo_coord_df with censo_hogar_df['ingr_hog'] and censo_persona_df on 'llaveviv', 'hogar', 'npersona', 'provincia', 'distrito', 'correg', 'area', 'lugar_pob', 'barriada'
personas_df = personas_df.merge(
    censo_hogar_df[['llaveviv', 'hogar', 'provincia', 'correg', 'area', 'ingr_hog']], 
    on=['llaveviv', 'hogar', 'provincia', 'correg', 'area'], 
    how='left'
)

In [149]:
personas_df.columns

Index(['p03_edad', 'p14_escu', 'p02_sexo', 'p14_asistio', 'pea_nea',
       'p18_ocup_cod4', 'ingr_per', 'p17e_motivo', 'llaveviv', 'hogar',
       'npersona', 'provincia', 'distrito', 'correg', 'area', 'lugar_pob',
       'barriada', 'p223_jubi', 'worker', 'unemployed', 'student_worker',
       'student', 'retired', 'inactive', 'low_income_worker',
       'high_income_worker', 'pension_retiree', 'no_pension_retiree', 'age',
       'sexo', 'ingr_hog'],
      dtype='object')

In [150]:
personas_df[['llaveviv', 'hogar',
       'npersona', 'provincia', 'distrito', 'correg', 'area', 'lugar_pob']]

,llaveviv,hogar,npersona,provincia,distrito,correg,area,lugar_pob
0,0000001,1,1,01,01,01,1,011
1,0000004,1,1,01,01,01,1,011
2,0000006,1,1,01,01,01,1,011
3,0000007,1,1,01,01,01,1,011
4,0000008,1,1,01,01,01,1,011
...,...,...,...,...,...,...,...,...
4064775,1595464,1,4,13,09,09,2,005
4064776,1595464,1,5,13,09,09,2,005
4064777,1595465,1,1,13,09,09,2,005
4064778,1595465,1,2,13,09,09,2,005


In [151]:
personas_df["cod_corr"] = personas_df["provincia"] + personas_df["distrito"] + personas_df["correg"]
personas_df["cod_prov"] = personas_df["provincia"]

In [152]:
# Create income quintiles based on household income
# First, handle any missing values
personas_df['ingr_hog'] = pd.to_numeric(personas_df['ingr_hog'], errors='coerce').fillna(0)

# Create quintiles (5 groups)
# Use qcut to create equal-sized groups, handling duplicates
personas_df['income_quintile'] = pd.qcut(
    personas_df['ingr_hog'], 
    q=5, 
    labels=['Q1', 'Q2', 'Q3', 'Q4', 'Q5'],
    duplicates='drop'
)

# Fill any NaN values (from duplicates) with 'Q3' as default
personas_df['income_quintile'] = personas_df['income_quintile'].fillna('Q3')

print("Income quintile distribution:")
print(personas_df['income_quintile'].value_counts())


Income quintile distribution:
income_quintile
Q2    813449
Q1    813239
Q3    812858
Q4    812659
Q5    812575
Name: count, dtype: int64


In [153]:
# Aggregate data by municipality, age, and sex (NO quintile for performance)
# This structure allows frontend to:
# - Filter by age (from slider)
# - Sum across sex for totals (default view)
# - Or filter by sex for breakdowns if needed

aggregated = personas_df.groupby([
    'cod_corr', 
    'cod_prov', 
    'age', 
    'sexo'
], observed=True).agg({
    'low_income_worker': 'sum',
    'high_income_worker': 'sum',
    'unemployed': 'sum',
    'inactive': 'sum',
    'student': 'sum',
    'student_worker': 'sum',
    'retired': 'sum',  # Combined retired (no separate pension categories)
    'npersona': 'count'  # total count of people in this group
}).reset_index()

# Rename count column
aggregated.rename(columns={'npersona': 'total_count'}, inplace=True)

print(f"Aggregated data shape: {aggregated.shape}")
print(f"\nSample of aggregated data:")
print(aggregated.head(10))


Aggregated data shape: (125545, 12)

Sample of aggregated data:
  cod_corr cod_prov age    sexo  low_income_worker  high_income_worker  \
0   010101       01   0  Hombre                  0                   0   
1   010101       01   0   Mujer                  0                   0   
2   010101       01   1  Hombre                  0                   0   
3   010101       01   1   Mujer                  0                   0   
4   010101       01   2  Hombre                  0                   0   
5   010101       01   2   Mujer                  0                   0   
6   010101       01   3  Hombre                  0                   0   
7   010101       01   3   Mujer                  0                   0   
8   010101       01   4  Hombre                  0                   0   
9   010101       01   4   Mujer                  0                   0   

   unemployed  inactive  student  student_worker  retired  total_count  
0           0        86        0               0

In [159]:
# Verify the aggregation structure
print("Age range:", aggregated['age'].min(), "to", aggregated['age'].max())
print("Unique municipalities (cod_corr):", aggregated['cod_corr'].nunique())
print("Sex values:", aggregated['sexo'].unique())
print("\nExample: Data for one municipality at age 25:")
print(aggregated[(aggregated['cod_corr'] == aggregated['cod_corr'].iloc[0]) & 
                 (aggregated['age'] == 25)].head())


Age range: 0 to No declarada
Unique municipalities (cod_corr): 699
Sex values: ['Hombre', 'Mujer']
Categories (2, object): ['Hombre' < 'Mujer']

Example: Data for one municipality at age 25:
   cod_corr cod_prov age    sexo  low_income_worker  high_income_worker  \
50   010101       01  25  Hombre                 27                  21   
51   010101       01  25   Mujer                 25                  17   

    unemployed  inactive  student  student_worker  retired  total_count  
50           2         5        1               6        0           56  
51           2        21        1              10        0           66  


In [160]:
# Create a compact version for the frontend
# Use shorter column names and only essential columns to reduce file size
print("Creating compact version...")
print(f"Original data shape: {aggregated.shape}")

aggregated_compact = aggregated[[
    'cod_corr', 'cod_prov', 'age', 'sexo',
    'low_income_worker', 'high_income_worker', 'unemployed', 'inactive',
    'student', 'student_worker', 'retired', 'total_count'
]].copy()

# Convert age to integer, filter out invalid ages BEFORE renaming
aggregated_compact['age'] = pd.to_numeric(aggregated_compact['age'], errors='coerce')
aggregated_compact = aggregated_compact[aggregated_compact['age'].notna()]
aggregated_compact['age'] = aggregated_compact['age'].astype(int)

# Filter to reasonable age range (3-70 as per requirements)
# Make sure to include all ages including 61-70
aggregated_compact = aggregated_compact[
    (aggregated_compact['age'] >= 3) & 
    (aggregated_compact['age'] <= 70) &
    (aggregated_compact['age'].notna())
]

# Rename to shorter names to reduce JSON size
aggregated_compact.columns = [
    'c',      # cod_corr
    'p',      # cod_prov  
    'a',      # age
    's',      # sexo
    'liw',    # low_income_worker
    'hiw',    # high_income_worker
    'u',      # unemployed
    'i',      # inactive
    'st',     # student
    'sw',     # student_worker
    'r',      # retired
    'tc'      # total_count
]

# Convert all numeric columns to int32 to save space (instead of int64)
numeric_cols = ['a', 'liw', 'hiw', 'u', 'i', 'st', 'sw', 'r', 'tc']
for col in numeric_cols:
    aggregated_compact[col] = aggregated_compact[col].astype('int32')

print(f"Compact data shape: {aggregated_compact.shape}")
print(f"Rows removed: {aggregated.shape[0] - aggregated_compact.shape[0]:,}")
print(f"Size reduction: {((1 - aggregated_compact.shape[0] / aggregated.shape[0]) * 100):.1f}%")

# Save compact version with minimal formatting
print("\nSaving compact version...")
aggregated_compact.to_json('data/aggregated_compact.json', orient='records', indent=None)

# Check file size
import os
file_size_mb = os.path.getsize('data/aggregated_compact.json') / (1024 * 1024)
print(f"Compact JSON file size: {file_size_mb:.2f} MB")

if file_size_mb > 100:
    print("\n⚠️  WARNING: File is still large (>100MB).")
    print("   Consider splitting by age ranges or using a different format.")
    print("   For now, the browser may still struggle with this file size.")
else:
    print("✓ File size is reasonable for browser loading")

# Show sample of compact data
print("\nSample of compact data:")
print(aggregated_compact.head(3))

# Since the file is still too large, split by age ranges for on-demand loading
print("\n" + "="*60)
print("Creating age-range split files (for on-demand loading)...")
print("="*60)

age_ranges = [
    (3, 20, 'ages_3_20'),
    (21, 40, 'ages_21_40'),
    (41, 60, 'ages_41_60'),
    (61, 70, 'ages_61_70')
]

total_size = 0
for min_age, max_age, filename in age_ranges:
    age_data = aggregated_compact[(aggregated_compact['a'] >= min_age) & (aggregated_compact['a'] <= max_age)]
    filepath = f'data/{filename}.json'
    age_data.to_json(filepath, orient='records', indent=None)
    file_size = os.path.getsize(filepath) / (1024 * 1024)
    total_size += file_size
    print(f"  {filename}.json: {age_data.shape[0]:,} rows, {file_size:.2f} MB")

print(f"\nTotal size of split files: {total_size:.2f} MB")
print(f"Original compact file: {file_size_mb:.2f} MB")

if total_size < 200:
    print("\n✓ Split files are manageable! Frontend can load age ranges on demand.")
else:
    print("\n⚠️  Split files are still large. Consider further optimization.")
    



Creating compact version...
Original data shape: (125545, 12)
Compact data shape: (92818, 12)
Rows removed: 32,727
Size reduction: 26.1%

Saving compact version...
Compact JSON file size: 8.85 MB
✓ File size is reasonable for browser loading

Sample of compact data:
        c   p  a       s  liw  hiw  u   i  st  sw  r  tc
6  010101  01  3  Hombre    0    0  0  78   0   0  0  78
7  010101  01  3   Mujer    0    0  0  73   0   0  0  73
8  010101  01  4  Hombre    0    0  0  58  12   0  0  70

Creating age-range split files (for on-demand loading)...
  ages_3_20.json: 24,688 rows, 2.35 MB
  ages_21_40.json: 27,369 rows, 2.62 MB
  ages_41_60.json: 27,334 rows, 2.60 MB
  ages_61_70.json: 13,427 rows, 1.27 MB

Total size of split files: 8.85 MB
Original compact file: 8.85 MB

✓ Split files are manageable! Frontend can load age ranges on demand.


In [158]:
# Example: Show how frontend would get totals (summing across sex and quintile)
# This is what the map would use by default

# Filter by age (e.g., age 25)
age_example = 5
filtered_by_age = aggregated[aggregated['age'] == age_example]

# Sum across sex and income_quintile to get totals per municipality
totals_by_municipality = filtered_by_age.groupby(['cod_corr', 'age']).agg({
    'low_income_worker': 'sum',
    'high_income_worker': 'sum',
    'unemployed': 'sum',
    'inactive': 'sum',
    'student_worker': 'sum',
    'student': 'sum',
    'total_count': 'sum'
}).reset_index()

print(f"Example: Totals for age {age_example} (summed across all sexes and quintiles):")
print(totals_by_municipality.head(10))
print(f"\nTotal municipalities with data for age {age_example}: {len(totals_by_municipality)}")


Example: Totals for age 5 (summed across all sexes and quintiles):
  cod_corr age  low_income_worker  high_income_worker  unemployed  inactive  \
0   010101   0                  0                   0           0         0   
1   010101   1                  0                   0           0         0   
2   010101   2                  0                   0           0         0   
3   010101   3                  0                   0           0         0   
4   010101   4                  0                   0           0         0   
5   010101   5                  0                   0           0        68   
6   010101   6                  0                   0           0         0   
7   010101   7                  0                   0           0         0   
8   010101   8                  0                   0           0         0   
9   010101   9                  0                   0           0         0   

   student_worker  student  total_count  
0               0    

C:\Users\lopez\AppData\Local\Temp\ipykernel_17552\4257694691.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  totals_by_municipality = filtered_by_age.groupby(['cod_corr', 'age']).agg({
